# 00. Data Preparation

Converts all raw supplementary tables and result files into clean DataFrames.
Outputs are saved to `../data/processed/` for use in downstream analysis notebooks.

**Source data:**
- Supp. Table S1 (strain metadata + NCBI accessions)
- Supp. Table S2 (phenotypic AMR — 79 isolates)
- Supp. Table S3 (virulence gene profile — 250+ genes)
- Supp. Table S4 (MGE profile)
- ANI matrix (FastANI results)
- VF heatmap matrix (Figure 3 source data)

**Reference:** Lee et al. (2023), *Frontiers in Microbiology*, 14:1175304

In [1]:
# ── dependencies ──────────────────────────────────────────────
# pip install pandas openpyxl

import pandas as pd
import re
from pathlib import Path

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(parents=True, exist_ok=True)

print('Paths OK')
print(f'  raw      : {RAW.resolve()}')
print(f'  processed: {PROC.resolve()}')

Paths OK
  raw      : C:\Users\SAMSUNG\Desktop\amr_genomics_aeromonas\data\raw
  processed: C:\Users\SAMSUNG\Desktop\amr_genomics_aeromonas\data\processed


---
## 1. Supp. Table S1 — strain metadata

In [2]:
# ── S1B: 22 isolates + NCBI accession numbers ─────────────────
s1b = pd.read_excel(RAW / 'Table_1.xlsx', sheet_name='Table_S1B',
                    header=1, usecols=[0, 1])
s1b.columns = ['strain_full', 'accession']

# strip extra whitespace
s1b = s1b.dropna(subset=['strain_full'])
s1b['strain_full'] = s1b['strain_full'].str.strip()
s1b['accession']   = s1b['accession'].str.strip()

# parse species and strain_id from 'Aeromonas salmonicida OY59' format
def parse_strain(name):
    parts = name.split()
    # 'Aeromonas species strain_id' → 3 tokens minimum
    if len(parts) >= 3:
        species  = ' '.join(parts[:2])   # e.g. 'Aeromonas salmonicida'
        strain_id = parts[2]              # e.g. 'OY59'
        short_sp  = parts[1]             # e.g. 'salmonicida'
    else:
        species   = name
        strain_id = name
        short_sp  = name
    return pd.Series([species, short_sp, strain_id])

s1b[['species_full', 'species_short', 'strain_id']] = \
    s1b['strain_full'].apply(parse_strain)

s1b.to_csv(PROC / 'strain_metadata.csv', index=False)
print(f'strain_metadata.csv — {len(s1b)} rows')
s1b.head()

strain_metadata.csv — 22 rows


,strain_full,accession,species_full,species_short,strain_id
1,Aeromonas salmonicida OY59,JAOPLB000000000,Aeromonas salmonicida,salmonicida,OY59
2,Aeromonas salmonicida OY56,JAOPLC000000000,Aeromonas salmonicida,salmonicida,OY56
3,Aeromonas salmonicida SL21,JAOPLD000000000,Aeromonas salmonicida,salmonicida,SL21
4,Aeromonas salmonicida SL19,JAOPLE000000000,Aeromonas salmonicida,salmonicida,SL19
5,Aeromonas rivipollensis SC42,JAOPLF000000000,Aeromonas rivipollensis,rivipollensis,SC42


---
## 2. Supp. Table S2 — phenotypic AMR (79 isolates)

In [3]:
# Excel structure (Table2.XLSX — user-updated file):
#   Row 0: title
#   Row 1: empty
#   Row 2: antibiotic class headers (Beta-lactams … MDR)
#   Row 3: antibiotic names — MDR cell is NaN here
#   Row 4+: data, MDR column = 1 (MDR) or 0 (non-MDR)

# ── read antibiotic class row ─────────────────────────────────
class_row = pd.read_excel(RAW / 'Table_2.XLSX', sheet_name='Table_S2',
                          header=None, skiprows=2, nrows=1)
class_row = class_row.ffill(axis=1)
abx_classes = class_row.iloc[0, 4:].tolist()   # includes 'MDR' at end

# ── main data ─────────────────────────────────────────────────
s2_raw = pd.read_excel(RAW / 'Table_2.XLSX', sheet_name='Table_S2',
                       header=3)

s2_raw.columns = (
    ['source', 'year', 'isolate_id', 'species_id'] +
    list(s2_raw.columns[4:])
)
s2_raw = s2_raw.dropna(subset=['isolate_id'])

# strip newlines from column names
s2_raw.columns = [
    c.replace('\n', ' ').strip() if isinstance(c, str) else c
    for c in s2_raw.columns
]

# last column reads as 'Unnamed: 19' — rename to MDR
cols = list(s2_raw.columns)
cols[-1] = 'MDR'
s2_raw.columns = cols

# antibiotic columns = cols 4 to -1
abx_cols = list(s2_raw.columns[4:-1])
# antibiotic → class lookup (exclude trailing 'MDR' entry)
abx_class_map = dict(zip(abx_cols, abx_classes[:-1]))

print('Raw shape:', s2_raw.shape)   # expect (79, 20)
print('MDR value counts:', s2_raw['MDR'].value_counts().to_dict())  # {1: 45, 0: 34}
print(f'MDR rate: {(s2_raw["MDR"]==1).mean()*100:.1f}%')           # ~57%
print()
print('Antibiotic → class:')
for k, v in abx_class_map.items():
    print(f'  {str(k):<42} → {v}')
print()
print('Example cell:', s2_raw.iloc[0, 4])   # '0 (R)'


Raw shape: (79, 20)
MDR value counts: {1.0: 45, 0.0: 34}
MDR rate: 57.0%

Antibiotic → class:
  Ampicillin AMP10                           → Beta-lactams
  Mecillinam MEL10                           → Beta-lactams
  Cefotaxime CTX30                           → Cephalosporins 
  ceftriaxone CRO30                          → Cephalosporins 
  Ciprofloxacin CIP1                         → Quinolones
  Oxolinic acid OA                           → Quinolones
  Doxycycline DO30                           → Tetracylines 
  Tetracycline TE30                          → Tetracylines 
  Imipenem IPM10                             → Carbapenemems
  Meropenem MEM10                            → Carbapenemems
  Gentamycin  GM10                           → Aminoglycosides 
  Tobramycin TOB30                           → Aminoglycosides 
  Erythromycin  EM15                         → Macrolides
  Florfenicol  FEC30                         → Amphenicols
  Trimethoprim/sulfamethoxazole  STX 25      → Amphenic

In [4]:
def parse_abx_cell(val):
    """Extract (zone_diameter, phenotype) from '32 (S)' format."""
    if pd.isna(val):
        return None, None
    m = re.match(r'^(\d+)\s*\(([RIS])\)$', str(val).strip())
    if m:
        return int(m.group(1)), m.group(2)
    return None, None

meta_cols = ['source', 'year', 'isolate_id', 'species_id']
records = []

for _, row in s2_raw.iterrows():
    base = {col: row[col] for col in meta_cols}
    base['MDR'] = int(row['MDR'])   # 1 or 0
    for abx in abx_cols:
        zone, phenotype = parse_abx_cell(row[abx])
        records.append({
            **base,
            'antibiotic'   : abx,
            'abx_class'    : abx_class_map.get(abx, 'Unknown'),
            'zone_diameter': zone,
            'phenotype'    : phenotype
        })

s2_long = pd.DataFrame(records)
s2_long.to_csv(PROC / 'amr_phenotype_long.csv', index=False)
print(f'amr_phenotype_long.csv — {len(s2_long)} rows')   # expect 1185
print(f'Unique isolates: {s2_long["isolate_id"].nunique()}')   # 79
print(f'Unique antibiotics: {s2_long["antibiotic"].nunique()}')   # 15
s2_long.head(4)


amr_phenotype_long.csv — 1185 rows
Unique isolates: 77
Unique antibiotics: 15


,source,year,isolate_id,species_id,MDR,antibiotic,abx_class,zone_diameter,phenotype
0,Retail sushi,2015.0,SU2,A. salmonicida,0,Ampicillin AMP10,Beta-lactams,0,R
1,Retail sushi,2015.0,SU2,A. salmonicida,0,Mecillinam MEL10,Beta-lactams,32,S
2,Retail sushi,2015.0,SU2,A. salmonicida,0,Cefotaxime CTX30,Cephalosporins,44,S
3,Retail sushi,2015.0,SU2,A. salmonicida,0,ceftriaxone CRO30,Cephalosporins,37,S


In [5]:
# wide format: isolate × antibiotic = phenotype
s2_wide = s2_long.pivot_table(
    index=['source', 'year', 'isolate_id', 'species_id', 'MDR'],
    columns='antibiotic',
    values='phenotype',
    aggfunc='first'
).reset_index()
s2_wide.columns.name = None

s2_wide.to_csv(PROC / 'amr_phenotype_wide.csv', index=False)
print(f'amr_phenotype_wide.csv — {len(s2_wide)} rows × {len(s2_wide.columns)} cols')
# expect: 79 rows × 20 cols (4 meta + MDR + 15 antibiotics)

# resistance rate per antibiotic
resist_rate = {
    abx: (s2_wide[abx] == 'R').mean() * 100
    for abx in abx_cols if abx in s2_wide.columns
}
print('\nResistance rate (%) per antibiotic:')
for k, v in sorted(resist_rate.items(), key=lambda x: -x[1]):
    print(f'  {str(k):<45} {v:.1f}%')


amr_phenotype_wide.csv — 79 rows × 20 cols

Resistance rate (%) per antibiotic:
  Ampicillin AMP10                              100.0%
  Erythromycin  EM15                            57.0%
  Florfenicol  FEC30                            48.1%
  Oxolinic acid OA                              21.5%
  Imipenem IPM10                                10.1%
  Cefotaxime CTX30                              2.5%
  Meropenem MEM10                               1.3%
  Mecillinam MEL10                              0.0%
  ceftriaxone CRO30                             0.0%
  Ciprofloxacin CIP1                            0.0%
  Doxycycline DO30                              0.0%
  Tetracycline TE30                             0.0%
  Gentamycin  GM10                              0.0%
  Tobramycin TOB30                              0.0%
  Trimethoprim/sulfamethoxazole  STX 25         0.0%


---
## 3. Supp. Table S3 — virulence gene profile

In [8]:
# ── Table S3 structure ────────────────────────────────────────
# Row 0: title
# Row 1: empty
# Row 2: column headers (VFclass, VF name, gene, strain names)
#         col 5 = A449 plasmid (Unnamed) — data always NaN, drop it
# Row 3: accession numbers (skip)
# Row 4+: data

# read row 2 manually to get clean strain names
header_row = pd.read_excel(RAW / 'Table_3.XLSX', sheet_name='Table_S3',
                           header=None, skiprows=2, nrows=1)
raw_headers = header_row.iloc[0].tolist()

# fix headers:
# col 0-2: meta, col 3: ATCC7966, col 4: A449_chr, col 5: A449_plasmid(drop), col 6+: isolates
clean_headers = []
for j, h in enumerate(raw_headers):
    if j == 0: clean_headers.append('VFclass')
    elif j == 1: clean_headers.append('VF_name')
    elif j == 2: clean_headers.append('gene')
    elif j == 3: clean_headers.append('ATCC7966')        # reference 1
    elif j == 4: clean_headers.append('A449')            # reference 2
    elif j == 5: clean_headers.append('DROP')            # A449 plasmid — always NaN
    else:
        # extract strain ID from 'A. salmonicida OY59' → 'OY59'
        h_str = str(h).strip() if pd.notna(h) else ''
        strain_id = h_str.split()[-1] if h_str else f'col{j}'
        clean_headers.append(strain_id)

print('Clean headers:')
for j, h in enumerate(clean_headers):
    print(f'  col {j}: {h}')


Clean headers:
  col 0: VFclass
  col 1: VF_name
  col 2: gene
  col 3: ATCC7966
  col 4: A449
  col 5: DROP
  col 6: A533
  col 7: A537
  col 8: SU4
  col 9: A539
  col 10: SU3
  col 11: OY1
  col 12: SC42
  col 13: OY52
  col 14: SU6
  col 15: SU58-3
  col 16: SL22
  col 17: LJP308
  col 18: SL19
  col 19: SL21
  col 20: OY56
  col 21: OY59
  col 22: SC45
  col 23: SU2
  col 24: LJP441


In [9]:
# ── read data from row 4 onward ────────────────────────────────
s3_data = pd.read_excel(RAW / 'Table_3.XLSX', sheet_name='Table_S3',
                        header=None, skiprows=4)
s3_data.columns = clean_headers

# drop the A449 plasmid column (always NaN)
s3_data = s3_data.drop(columns=['DROP'])

# forward-fill VFclass and VF_name (merged cells)
s3_data['VFclass'] = s3_data['VFclass'].ffill()
s3_data['VF_name'] = s3_data['VF_name'].ffill()

# drop rows with no gene name
s3_data = s3_data.dropna(subset=['gene'])
s3_data['gene'] = s3_data['gene'].astype(str).str.strip()
s3_data = s3_data[s3_data['gene'] != 'nan']

# strain columns = everything after VFclass, VF_name, gene
strain_cols = list(s3_data.columns[3:])

print(f'Shape: {s3_data.shape}')
print(f'VF classes: {s3_data["VFclass"].unique()}')
print(f'Strain columns ({len(strain_cols)}): {strain_cols}')
s3_data[['VFclass', 'VF_name', 'gene'] + strain_cols[:3]].head(6)

Shape: (287, 24)
VF classes: ['Adherence' 'Secretion system' 'Toxin' 'Endotoxin' 'O-antigen'
 'Antiphagocytosis' 'Immune evasion' 'Glycosylation system'
 'Serum resistance' 'Stress adaptation' 'Iron uptake' 'Biofilm formation'
 'Copper uptake']
Strain columns (21): ['ATCC7966', 'A449', 'A533', 'A537', 'SU4', 'A539', 'SU3', 'OY1', 'SC42', 'OY52', 'SU6', 'SU58-3', 'SL22', 'LJP308', 'SL19', 'SL21', 'OY56', 'OY59', 'SC45', 'SU2', 'LJP441']


,VFclass,VF_name,gene,ATCC7966,A449,A533
0,Adherence,Flp type IV pili,flp1,AHA_1450,ASA_2915,-
1,Adherence,Flp type IV pili,flpA,AHA_1451,ASA_2914,-
2,Adherence,Flp type IV pili,flpB,AHA_1452,ASA_2913*,-
3,Adherence,Flp type IV pili,flpC,AHA_1453,ASA_2912,-
4,Adherence,Flp type IV pili,flpD,AHA_1454,ASA_2911,-
5,Adherence,Flp type IV pili,flpE,AHA_1455,ASA_2910,-


In [11]:
# ── convert locus tags → binary (present=1, absent=0) ─────────
strain_cols = list(s3_data.columns[3:])

def to_binary(val):
    """Locus tag present → 1, '-' or NaN → 0."""
    if pd.isna(val):
        return 0
    val = str(val).strip()
    if val == '-' or val == '':
        return 0
    return 1

s3_binary = s3_data[['VFclass', 'VF_name', 'gene']].copy()
for col in strain_cols:
    s3_binary[col] = s3_data[col].apply(to_binary)

s3_binary.to_csv(PROC / 'virulence_genes_binary.csv', index=False)
print(f'virulence_genes_binary.csv — {len(s3_binary)} genes × {len(strain_cols)} strains')

# preview
s3_binary[['VFclass', 'VF_name', 'gene'] + strain_cols[:4]].head(8)

virulence_genes_binary.csv — 287 genes × 21 strains


,VFclass,VF_name,gene,ATCC7966,A449,A533,A537
0,Adherence,Flp type IV pili,flp1,1,1,0,0
1,Adherence,Flp type IV pili,flpA,1,1,0,0
2,Adherence,Flp type IV pili,flpB,1,1,0,0
3,Adherence,Flp type IV pili,flpC,1,1,0,0
4,Adherence,Flp type IV pili,flpD,1,1,0,0
5,Adherence,Flp type IV pili,flpE,1,1,0,0
6,Adherence,Flp type IV pili,flpF,1,1,0,0
7,Adherence,Flp type IV pili,flpG,1,1,0,0


In [12]:
# ── category-level summary: genes present per VFclass per strain
vf_summary = (
    s3_binary
    .groupby('VFclass')[strain_cols]
    .sum()                     # count genes present per category
    .T                         # transpose: strains as rows
)
vf_summary.index.name = 'strain'
vf_summary.to_csv(PROC / 'virulence_category_counts.csv')
print(f'virulence_category_counts.csv — {vf_summary.shape}')
vf_summary.head()

virulence_category_counts.csv — (21, 13)


VFclass,Adherence,Antiphagocytosis,Biofilm formation,Copper uptake,Endotoxin,Glycosylation system,Immune evasion,Iron uptake,O-antigen,Secretion system,Serum resistance,Stress adaptation,Toxin
strain,,,,,,,,,,,,,
ATCC7966,113,0,0,0,0,0,0,0,0,40,0,0,12
A449,146,0,0,0,0,0,0,0,0,30,0,0,5
A533,102,1,0,0,0,0,1,1,0,40,0,0,12
A537,99,1,0,0,0,0,0,2,0,38,0,0,12
SU4,94,0,0,0,0,0,0,0,0,16,1,1,3


---
## 4. Supp. Table S4 — MGE profile

In [16]:
# ── Table S4 structure ────────────────────────────────────────
# Row 0: title
# Row 1: empty
# Row 2: species names (merged cells, NaN for sub-cols)
# Row 3: strain IDs  ← actual column names to use
# Row 4: IS_family | subgroup | MGE_type headers
# Row 5+: data

# ── read strain IDs from row 3 ────────────────────────────────
strain_id_row = pd.read_excel(RAW / 'Table_4.XLSX', sheet_name='Table_S4',
                              header=None, skiprows=3, nrows=1)
# col 0,1 = NaN (Families/Sub-groups), col 2 = 'StrainID', col 3+ = actual IDs
strain_ids = strain_id_row.iloc[0, 3:].tolist()   # 22 strain IDs
print('Strain IDs:', strain_ids)

# ── read data from row 4 onward ───────────────────────────────
s4_raw = pd.read_excel(RAW / 'Table_4.XLSX', sheet_name='Table_S4',
                       header=4)   # row 4 = IS_family, subgroup, MGE_type

# rename first 3 cols
s4_raw.columns = (
    ['IS_family', 'subgroup', 'MGE_type'] +
    list(s4_raw.columns[3:])
)

# replace col 3+ names with clean strain IDs from row 3
all_cols = list(s4_raw.columns)
all_cols[3:] = [str(sid).strip() for sid in strain_ids]
s4_raw.columns = all_cols

# forward-fill IS_family and subgroup (merged cells)
s4_raw['IS_family'] = s4_raw['IS_family'].ffill()
s4_raw['subgroup']  = s4_raw['subgroup'].ffill()

# drop rows where MGE_type is NaN or non-data values
s4_raw = s4_raw.dropna(subset=['MGE_type'])
s4_raw = s4_raw[~s4_raw['MGE_type'].isin(['StrainID', 'Type of MGE'])]
s4_raw = s4_raw.reset_index(drop=True)

strain_cols_s4 = list(s4_raw.columns[3:])

print(f'\nRaw shape: {s4_raw.shape}')   # expect ~54 rows × 25 cols
print(f'MGE types: {s4_raw["MGE_type"].unique()}')
print(f'Strain columns ({len(strain_cols_s4)}): {strain_cols_s4}')
s4_raw[['IS_family', 'subgroup', 'MGE_type'] + strain_cols_s4[:4]].head(6)


Strain IDs: ['SU4', 'A533', 'A536', 'A537', 'A539', 'SU3', 'SU9', 'SU15', 'SC42', 'OY1', 'OY52', 'SU6', 'SL22', 'SU58-3', 'LP308', 'SU2', 'SL19', 'SL21', 'SC45', 'OY56', 'OY59', 'LP441']

Raw shape: (52, 25)
MGE types: ['IS' 'Composite transposon (cn)' 'Miniature Inverted Repeat (MITE)' 'IS5'
 'Unit transposon (Tn)']
Strain columns (22): ['SU4', 'A533', 'A536', 'A537', 'A539', 'SU3', 'SU9', 'SU15', 'SC42', 'OY1', 'OY52', 'SU6', 'SL22', 'SU58-3', 'LP308', 'SU2', 'SL19', 'SL21', 'SC45', 'OY56', 'OY59', 'LP441']


,IS_family,subgroup,MGE_type,SU4,A533,A536,A537
0,IS1595,NaN,IS,ISKpn3,NaN,NaN,NaN
1,IS1595,ISPna2,IS,NaN,NaN,NaN,NaN
2,IS1595,ISPna2,IS,NaN,NaN,NaN,ISAhy1
3,IS3,IS2,IS,ISAs17,NaN,NaN,ISAs17
4,IS3,IS2,IS,NaN,NaN,NaN,NaN
5,IS3,IS3,IS,NaN,ISAs33,ISAs33,NaN


In [17]:
# ── convert to binary + extract copy number where present ─────

def parse_mge_cell(val):
    """Return (present: 0/1, copy_number: int).
    Handles: NaN → (0,0), 'ISAs31' → (1,1), 'ISAs31 (4)' → (1,4)
    """
    if pd.isna(val):
        return 0, 0
    val = str(val).strip()
    if val == '':
        return 0, 0
    m = re.search(r'\((\d+)\)', val)   # look for copy number in parentheses
    copy_n = int(m.group(1)) if m else 1
    return 1, copy_n

# binary presence
s4_binary = s4_raw[['IS_family', 'subgroup', 'MGE_type']].copy()
s4_copies = s4_raw[['IS_family', 'subgroup', 'MGE_type']].copy()

for col in strain_cols_s4:
    presence, copies = zip(*s4_raw[col].apply(parse_mge_cell))
    s4_binary[col] = list(presence)
    s4_copies[col] = list(copies)

s4_binary.to_csv(PROC / 'mge_binary.csv', index=False)
s4_copies.to_csv(PROC / 'mge_copy_number.csv', index=False)
print(f'mge_binary.csv     — {s4_binary.shape}')
print(f'mge_copy_number.csv — {s4_copies.shape}')
s4_binary[['IS_family', 'subgroup', 'MGE_type'] + strain_cols_s4[:5]].head(8)

mge_binary.csv     — (52, 25)
mge_copy_number.csv — (52, 25)


,IS_family,subgroup,MGE_type,SU4,A533,A536,A537,A539
0,IS1595,NaN,IS,1,0,0,0,0
1,IS1595,ISPna2,IS,0,0,0,0,0
2,IS1595,ISPna2,IS,0,0,0,1,0
3,IS3,IS2,IS,1,0,0,1,0
4,IS3,IS2,IS,0,0,0,0,0
5,IS3,IS3,IS,0,1,1,0,0
6,IS3,IS3,IS,0,0,0,1,0
7,IS3,IS3,IS,0,0,0,0,1


In [18]:
# ── separate IS elements vs transposons ───────────────────────
is_only = s4_binary[s4_binary['MGE_type'] == 'IS']
tn_only = s4_binary[s4_binary['MGE_type'].str.contains('transposon', case=False, na=False)]

is_only.to_csv(PROC / 'is_elements_binary.csv', index=False)
tn_only.to_csv(PROC / 'transposons_binary.csv', index=False)

print(f'IS elements : {len(is_only)} rows')
print(f'Transposons : {len(tn_only)} rows')

# IS count per strain
is_count = is_only[strain_cols_s4].sum().sort_values(ascending=False)
print('\nIS element count per strain (top 5):')
print(is_count.head())

IS elements : 39 rows
Transposons : 9 rows

IS element count per strain (top 5):
SU4      14
A539     11
SL21      9
LP441     9
A537      8
dtype: int64


---
## 5. ANI matrix

In [19]:
# ── clustered matrix (Figure 2 order) ─────────────────────────
ani_clustered = pd.read_csv(
    RAW / 'ANIclustermap_matrix.tsv',
    sep='\t', header=0, index_col=None
)

# first column = strain names (set as index)
strain_names = ani_clustered.iloc[:, 0].tolist() if ani_clustered.columns[0] not in ani_clustered.columns[1:] else None

# the TSV has no index column — rows are in same order as columns
# set index from column names
ani_clustered.index = ani_clustered.columns.tolist()

# confirm it's square
print(f'ANI matrix shape: {ani_clustered.shape}')  # should be 30×30
print(f'Diagonal values (should all be 100.0):')
diag_vals = [ani_clustered.iloc[i, i] for i in range(min(5, len(ani_clustered)))]
print(diag_vals)

ani_clustered.to_csv(PROC / 'ani_matrix_clustered.csv')
print(f'\nani_matrix_clustered.csv saved — {ani_clustered.shape}')

ANI matrix shape: (30, 30)
Diagonal values (should all be 100.0):
[np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(100.0)]

ani_matrix_clustered.csv saved — (30, 30)


In [20]:
# ── raw matrix (original FastANI order) ───────────────────────
ani_raw = pd.read_csv(
    RAW / 'fastani_matrix.tsv',
    sep='\t', header=0, index_col=None
)
ani_raw.index = ani_raw.columns.tolist()

ani_raw.to_csv(PROC / 'ani_matrix_raw.csv')
print(f'ani_matrix_raw.csv saved — {ani_raw.shape}')

ani_matrix_raw.csv saved — (30, 30)


---
## 6. Virulence heatmap matrix (Figure 3 source)

In [21]:
# ── category-level binary (strain × VF category) ──────────────
vf_heatmap = pd.read_excel(RAW / 'VF_profile_heatmap.xlsx',
                           sheet_name='Sheet1', index_col=0)

print(f'VF heatmap shape: {vf_heatmap.shape}')   # 27 strains × 18 VF categories
print(f'Columns: {list(vf_heatmap.columns)}')
print(f'Strains: {list(vf_heatmap.index)}')

vf_heatmap.to_csv(PROC / 'vf_heatmap_matrix.csv')
print(f'\nvf_heatmap_matrix.csv saved')
vf_heatmap

VF heatmap shape: (29, 18)
Columns: ['Msh type IV pili', 'Tap type IV pili', 'Flp type IV', 'Lateral flagella', 'Polar flagella', 'T2SS', 'T3SS', 'T6SS', 'aerA/act', 'ast', 'ahh1', 'hlyA', 'hemIII', 'th', 'RTX', 'toxA', 'capsule', 'O-antigen']
Strains: ['LJP308', 'A.piscicola_LMG_24783', 'SU58-3', 'SU6', 'SL22', 'A.bestiarum GA97_22', 'OY59', 'SU2', 'OY56', 'SL21', 'LJP441', 'SC45', 'SL19', 'A.salmonicida_O23A', 'SU4', 'A.caviae_1605_27183', 'A537', 'A.hydrophila_AH10', 'A.dhakensis_Aer_OnlF1', 'A536', 'A533', 'OY52', 'A.media_R1_18', 'SC42', 'OY1', 'A539', 'SU9', 'SU3', 'SU15']

vf_heatmap_matrix.csv saved


,Msh type IV pili,Tap type IV pili,Flp type IV,Lateral flagella,Polar flagella,T2SS,T3SS,T6SS,aerA/act,ast,ahh1,hlyA,hemIII,th,RTX,toxA,capsule,O-antigen
LJP308,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1
A.piscicola_LMG_24783,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,0,1
SU58-3,1,1,1,0,1,1,0,0,1,1,1,1,1,1,1,1,1,0
SU6,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,0,1,0
SL22,1,1,0,1,1,1,1,1,1,1,1,1,1,1,0,0,1,0
A.bestiarum GA97_22,1,1,0,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1
OY59,1,1,1,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1
SU2,1,1,1,0,1,1,0,1,1,0,1,1,1,1,1,0,1,1
OY56,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1
SL21,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,0


In [22]:
# ── column annotation: VF category → functional group ─────────
# directly from the R script (pheatmap annotation_col)
vf_col_annotation = pd.DataFrame({
    'VF_category': vf_heatmap.columns,
    'functional_group': (
        ['Adherence']    * 3 +   # Msh type IV, Tap type IV, Flp type IV
        ['Motility']     * 2 +   # Lateral flagella, Polar flagella
        ['Secretion']    * 3 +   # T2SS, T3SS, T6SS
        ['Toxin']        * 8 +   # aerA/act, ast, ahh1, hlyA, hemIII, th, RTX, toxA
        ['Immune evasion'] * 2   # capsule, O-antigen
    )
})
vf_col_annotation.to_csv(PROC / 'vf_col_annotation.csv', index=False)
print('vf_col_annotation.csv saved')
vf_col_annotation

vf_col_annotation.csv saved


,VF_category,functional_group
0,Msh type IV pili,Adherence
1,Tap type IV pili,Adherence
2,Flp type IV,Adherence
3,Lateral flagella,Motility
4,Polar flagella,Motility
5,T2SS,Secretion
6,T3SS,Secretion
7,T6SS,Secretion
8,aerA/act,Toxin
9,ast,Toxin


In [23]:
# ── row annotation: strain → species ──────────────────────────
# map strain names in VF heatmap to species
# order follows VF_profile_heatmap.xlsx row order
species_map = {
    'LJP308'                : 'A. piscicola',
    'A.piscicola_LMG_24783' : 'A. piscicola',
    'SU58-3'                : 'A. bestiarum',
    'SU6'                   : 'A. bestiarum',
    'SL22'                  : 'A. bestiarum',
    'A.bestiarum_GA97_22'   : 'A. bestiarum',
    'OY59'                  : 'A. salmonicida',
    'SU2'                   : 'A. salmonicida',
    'OY56'                  : 'A. salmonicida',
    'SL21'                  : 'A. salmonicida',
    'LJP441'                : 'A. salmonicida',
    'SC45'                  : 'A. salmonicida',
    'SL19'                  : 'A. salmonicida',
    'A.salmonicida_O23A'    : 'A. salmonicida',
    'SU4'                   : 'A. caviae',
    'A.caviae_1605_27183'   : 'A. caviae',
    'A537'                  : 'A. hydrophila',
    'A.hydrophila_AH10'     : 'A. hydrophila',
    'A.dhakensis_Aer_OnlF1' : 'A. dhakensis',
    'A536'                  : 'A. dhakensis',
    'A533'                  : 'A. dhakensis',
    'OY52'                  : 'A. media',
    'A.media_R1_18'         : 'A. media',
    'SC42'                  : 'A. rivipollensis',
    'OY1'                   : 'A. rivipollensis',
    'A539'                  : 'A. rivipollensis',
    'SU9'                   : 'A. rivipollensis',
}

vf_row_annotation = pd.DataFrame({
    'strain'  : list(species_map.keys()),
    'species' : list(species_map.values())
})
vf_row_annotation.to_csv(PROC / 'vf_row_annotation.csv', index=False)
print('vf_row_annotation.csv saved')
vf_row_annotation

vf_row_annotation.csv saved


,strain,species
0,LJP308,A. piscicola
1,A.piscicola_LMG_24783,A. piscicola
2,SU58-3,A. bestiarum
3,SU6,A. bestiarum
4,SL22,A. bestiarum
5,A.bestiarum_GA97_22,A. bestiarum
6,OY59,A. salmonicida
7,SU2,A. salmonicida
8,OY56,A. salmonicida
9,SL21,A. salmonicida


---
## 7. Summary — processed files

In [24]:
# ── final check: list all processed files ─────────────────────
print('Files saved to data/processed/:')
print()
for f in sorted(PROC.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>7.1f} KB')
print()
print('Data preparation complete. Ready for analysis notebooks.')

Files saved to data/processed/:

  amr_phenotype_long.csv                           92.6 KB
  amr_phenotype_wide.csv                            5.9 KB
  ani_matrix_clustered.csv                          9.2 KB
  ani_matrix_raw.csv                                9.2 KB
  is_elements_binary.csv                            2.4 KB
  mge_binary.csv                                    3.4 KB
  mge_copy_number.csv                               3.4 KB
  strain_metadata.csv                               1.8 KB
  transposons_binary.csv                            0.8 KB
  vf_col_annotation.csv                             0.3 KB
  vf_heatmap_matrix.csv                             1.4 KB
  vf_row_annotation.csv                             0.6 KB
  virulence_category_counts.csv                     0.9 KB
  virulence_genes_binary.csv                       21.8 KB

Data preparation complete. Ready for analysis notebooks.
